In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import datetime,timedelta
import json
import urllib.request
import matplotlib.pyplot as plt
import warnings
import psycopg2
warnings.filterwarnings("ignore")


analytics_mongo = client_db.get_database('analytics_db')

db = client.get_database('telematics_db')

db2 = client2.get_database('telematics_db')

wave=app_mongo.get_database('app_db')



In [ ]:
def loc_pings(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]
    
    try:
        url=GPS_API_BASE+"/gps/vehicles?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(int(vid),int(day_start),int(day_end))
        data=json.load(urllib.request.urlopen(url))
        df=pd.DataFrame(data['data'][str(int(vid))])
        
    except Exception as e:
 #       print(" loc pings issue >> " + str(e))
        df=pd.DataFrame()
    return df

In [ ]:
vehicle_devices=pd.read_csv("query_result_2020-12-22T17_30_50.696856+05_30.csv")    




In [ ]:
def vehicle_no_info(vehicle_id):
    try:
        data=loc_pings([vehicle_id,tm.time()-86400*5,tm.time()])
        data=data[['created','vehicleId']]
        data.sort_values('created',inplace=True)
        data.created=data.created/100
        data.created_before=data.created.shift(1)
        data['diff']=data.created-data.created_before
        data['diff']=data['diff']/60
        print(vehicle_id)
    except:
        data=pd.DataFrame()
    return data
        

In [ ]:
import multiprocessing
pool = multiprocessing.Pool(10)
results2=pool.map(vehicle_no_info,vehicle_devices['vehicle_devices → vehicle_id'][0:100])
pool.close()
pool.join()    

In [ ]:
data=pd.concat(results2)

In [ ]:
data.vehicleId.nunique()

In [ ]:
data.groupby('vehicleId').diff.describe(percentiles=[i/10 for i in range(0,10)])

In [ ]:
data[data['diff']>10].groupby('vehicleId').diff.count().mean()

In [ ]:
data[data['diff']>30].groupby('vehicleId').diff.count().mean()

In [ ]:
data[data['diff']>60].groupby('vehicleId').diff.count().mean()

In [ ]:
data[data['diff']>180].groupby('vehicleId').diff.count().mean()

In [ ]:
data[data['diff']>10].groupby('vehicleId').diff.count().median()

In [ ]:
data[data['diff']>30].groupby('vehicleId').diff.count().median()

In [ ]:
data[data['diff']>60].groupby('vehicleId').diff.count().median()

In [ ]:
data[data['diff']>180].groupby('vehicleId').diff.count().median()